# Sensitivity analyses: Supplement C cases and question format

Exploration notebook. Nothing here modifies the paper, `code/`, or `tables_and_figures/`.

* **Full-pipeline results** (sections 1–3) come from `run_variants.py`, which reruns `figuresHQ.ipynb` unchanged except that flagged races are dropped right after the data load. Those numbers are exactly what the paper would print under each exclusion.
* **Quick explorer** (section 4) recomputes simple RMSE summaries straight from `most_specific.xlsx` for any exclusion you choose. Use it to explore; it does **not** reproduce Table 8/9 logic (variable classification, year pairing), so confirm anything you'd cite with `run_variants.py`.

Regenerate: `python3 build_flags.py && python3 run_variants.py && python3 compare.py` (from this folder).

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from run_variants import VARIANTS
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 90)
RES = Path('results')
pd.DataFrame({v: d for v, (_, _, d) in VARIANTS.items()}, index=['description']).T

## 1. Headline numbers by variant
One row per prose macro behind a main claim; columns are variants. `base` should equal the committed `prose_numbers.tex`.

In [ ]:
hl = pd.read_csv(RES/'headline_by_variant.csv', index_col='macro')
hl

In [ ]:
# Every prose macro that differs from base in at least one variant
pn = pd.read_csv(RES/'prose_numbers_by_variant.csv', index_col='macro')
pn[pn.ne(pn['base'], axis=0).any(axis=1)]

## 2. Which exhibits change
`True` = the LaTeX table differs from base. Flattened text of each changed table, all variants stacked, is in `results/tables_text/`.

In [ ]:
tc = pd.read_csv(RES/'tables_changed.csv').set_index('table')
tc[tc.any(axis=1)]

In [ ]:
# Show one changed exhibit across variants (edit the name)
print((RES/'tables_text'/'office_rmse_progressive_race_exclusion.txt').read_text())

## 3. The flagged races
`flagged_races.csv`: one row per race × flag, with the race's CES-weighted error. See `build_flags.py` for each flag's definition.

In [ ]:
fl = pd.read_csv('flagged_races.csv')
fl = fl[fl.In_Analysis]
print(fl.groupby('Flag').agg(races=('Race_Key','nunique'),
      mean_abs_err=('Error', lambda e: e.abs().mean()),
      rmse=('Error', lambda e: np.sqrt((e**2).mean()))).round(2))

In [ ]:
# Largest errors among Supplement C / hand-fix flags
a1 = ['suppC_list_mismatch','suppC_nominee_bench','suppC_fusion','suppC_special','ces_manual_fix','returns_hand_fix']
fl[fl.Flag.isin(a1)].assign(abs_err=lambda d: d.Error.abs()).sort_values('abs_err', ascending=False).head(25)

## 4. Quick explorer (not paper-exact)
Drop any combination of flags, a benchmark threshold, or explicit race keys, and see office- and class-level RMSE for matching only and CES weights, plus each office's change from no exclusions.

In [ ]:
ms = pd.read_excel('../tables_and_figures/input_tables/most_specific.xlsx')
ms = ms[(ms.Validity_Scheme=='All') & ms.Error.notna()].copy()
ms['Race_Key'] = (ms.Year.astype(int).astype(str)+'|'+ms.State.str.upper()+'|'+ms.Variable+'|'
                  + ms.District.fillna('statewide').astype(str).replace({'nan':'statewide'}))
flags_all = pd.read_csv('flagged_races.csv')

def rmse(e): return float(np.sqrt(np.mean(np.square(e)))) if len(e) else np.nan

def quick_metrics(flags=(), max_benchmark=None, extra_keys=(), keep_offices=None):
    drop = set(flags_all.loc[flags_all.Flag.isin(flags), 'Race_Key']) | set(extra_keys)
    cc = ms.Class=='Candidate Choice'
    m = cc & ms.Race_Key.isin(drop)
    if max_benchmark is not None:
        m |= cc & (ms.Benchmark > max_benchmark)
    if keep_offices is not None:
        m |= cc & ~ms.Variable.isin(keep_offices)
    d = ms[~m]
    def summarize(frame, by):
        t = (frame[frame.Weighting_Method.isin(['CES-Unweighted','CES-Provided Weights'])]
             .groupby([by,'Weighting_Method']).Error.agg(rmse).unstack())
        t.columns = ['Weighted' if 'Provided' in c else 'Matching' for c in t.columns]
        t['N'] = frame[frame.Weighting_Method=='CES-Provided Weights'].groupby(by).size()
        return t[['N','Matching','Weighted']]
    office = summarize(d[d.Class=='Candidate Choice'], 'Variable')
    office0 = summarize(ms[ms.Class=='Candidate Choice'], 'Variable')
    office['dWeighted_vs_all'] = office.Weighted - office0.Weighted
    klass = summarize(d, 'Class')
    print(f'dropped {int(m.sum()/4)} candidate-choice estimates (per weighting method)')
    return office.round(2), klass.round(2)

office, klass = quick_metrics(flags=a1)
display(office); display(klass)

In [ ]:
# Examples
# quick_metrics(flags=['suppC_fusion'])
# quick_metrics(flags=a1 + ['uncontested_gt90'])
# quick_metrics(max_benchmark=95)
# quick_metrics(flags=['party_label_item'])           # named-candidate items only
# quick_metrics(extra_keys=['2012|NORTH CAROLINA|Attorney General|statewide'])

### 4b. Fragility: how many of the worst races does it take?
Drop the k candidate-choice races with the largest CES-weighted |error| and watch office RMSE. If a conclusion survives large k, it is not driven by a few cases.

In [ ]:
w = ms[(ms.Class=='Candidate Choice') & (ms.Weighting_Method=='CES-Provided Weights')]
worst = w.assign(a=w.Error.abs()).sort_values('a', ascending=False).Race_Key.tolist()
rows = []
for k in [0, 10, 25, 50, 100, 200, 400]:
    o, _ = quick_metrics(extra_keys=worst[:k])
    rows.append(o.Weighted.rename(f'drop top {k}'))
pd.concat(rows, axis=1).round(2)